# Phase 12 — Schrift-Tausch

Trennt die beiden Lesarten der Jacobi-Linse. Der Prompt bleibt token-identisch
bis auf die Locale-Hinweise; nur die **geforderte Ausgabeschrift** variiert.
Drei Arme (original / latein / placebo), Ausleseschicht **L35 vorregistriert**,
gepaarter Bootstrap über Korpus-Prompts.

Selbstversorgend — **frische Runtime**, dann nur diese Zelle. ~20–35 min.

In [ ]:
# === SCHRIFT-TAUSCH: haengt die Disposition an der SCHRIFT oder am WORT? ===
# Die Jacobi-Linse zeigt an Position 43 (' name') eine Fremdschrift-Disposition:
# Rang 1 von 18 an L27, L31 und L35, davon zwei Schichten NICHT selektiert.
# Der Konzept-Vorlauf hat die Frage aber offen gelassen - Sprachvokabular UND
# Lokalisierungs-Bezeichner spiken beide an Q, und beide liegen lexikalisch
# neben der Zeichenkette "local name". Die Mengen trennen die Hypothesen nicht.
#
# Dieser Versuch trennt sie. Der Prompt bleibt bis auf die Locale-Hinweise
# unveraendert - ' local' und ' name' und ihre gesamte Unembedding-Nachbarschaft
# sind in allen Armen identisch, jedes statische Token-Artefakt muss also
# invariant bleiben. Variiert wird nur die GEFORDERTE AUSGABESCHRIFT:
#   original  unveraendert
#   latein    nicht-lateinische Locale-Hinweise -> lateinische
#   placebo   nicht-lateinische -> ANDERE nicht-lateinische
# Der Placebo-Arm aendert gleich viel Text. Bricht er mit ein, war es die
# Bearbeitung und nicht die Schrift - dann trennt der Test nicht, und das
# Verdikt sagt das auch so.
#
# Vorregistriert, bevor Zahlen gesehen wurden:
#   Ausleseschicht L35 - die tiefste, ohne Blick auf Q gewaehlt. L31 und L27
#   nur sekundaer.
#   Zielgroesse Q/Median DERSELBEN Schicht und desselben Arms, pro Korpus-
#   Prompt einzeln - der Korpus-Prompt ist die Einheit fuer den gepaarten
#   Bootstrap. Absolute Massen sind zwischen Armen nicht vergleichbar.
#   Vorhersage Koeder-Lesart: latein bricht ein, placebo nicht.
#   Vorhersage Lokalisierungs-Lesart: beide bleiben wie das Original.
#
# Transport ueber die zentrale Differenz - der exakte Doppel-Rueckwaerts-Weg
# kostet 11 min je Korpus-Prompt, weil 30 der 40 Schichten gated DeltaNet im
# Torch-Fallback sind. Drei Quellschichten statt neun, dafuer drei Arme.
# Geschaetzt 20-35 min; die Zelle druckt die Zeit je Prompt mit.
#
# Vorregistrierung Nr. 29: KOEDER ~40%, UNSPEZIFISCH ~25%, LOKALISIERUNG ~20%,
# UNKLAR ~15%. Selbstversorgend, FRISCHE Runtime, einzige Zelle.

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, numpy as np
# ---------------- Selbstversorgung: Modell + Prompts sicherstellen ----------
import glob, json, gc
for _n in ("model_b","tok_b"):                  # Base-Reste aus Cell 28 raus
    if _n in globals():
        try: del globals()[_n]
        except Exception: pass
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError(("GPU nicht leer genug (%.1f GB frei, ~45 noetig): vermutlich "
        "belegt noch ein frueheres Modell den Speicher. Loesung: Laufzeit -> "
        "Sitzung neu starten, dann NUR diese Zelle ausfuehren.")%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
# ---------------- Protokoll und Abbildungen automatisch nach Drive ----------
# Der PDF-Export von Colab schneidet die Ausgabe unzuverlaessig ab. Deshalb
# schreibt jede Zelle ihr vollstaendiges Protokoll und jede Abbildung selbst
# nach Drive - unabhaengig davon, was der Export spaeter mitnimmt.
import sys, time
WC_RUN=globals().get("WC_RUN","lauf")
RUN_OUT="/content/drive/MyDrive/WeirdChat_Runs/%s_%s"%(WC_RUN,time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(RUN_OUT,exist_ok=True)
class _WCTee:
    _wc_tee=True
    def __init__(self,p,o): self.o=o; self.f=None; self.retarget(p)
    def retarget(self,p):
        try:
            if self.f: self.f.close()
        except Exception: pass
        try: self.f=open(p,"a",encoding="utf-8")
        except Exception: self.f=None
    def write(self,s):
        self.o.write(s)
        if self.f:
            try: self.f.write(s); self.f.flush()
            except Exception: pass
        return len(s)
    def flush(self):
        self.o.flush()
        if self.f:
            try: self.f.flush()
            except Exception: pass
    def isatty(self): return False
_wc_log=os.path.join(RUN_OUT,"protokoll.txt")
if getattr(sys.stdout,"_wc_tee",False): sys.stdout.retarget(_wc_log)
else: sys.stdout=_WCTee(_wc_log,sys.stdout)
try:
    import matplotlib.pyplot as _wcplt
    if not getattr(_wcplt,"_wc_patched",False):
        _wc_orig_show=_wcplt.show; _wc_fig=[0]
        def _wc_show(*a,**k):
            for _num in _wcplt.get_fignums():
                _wc_fig[0]+=1
                try:
                    _wcplt.figure(_num).savefig(os.path.join(RUN_OUT,"abb_%02d.png"%_wc_fig[0]),
                                                dpi=150,bbox_inches="tight")
                except Exception: pass
            return _wc_orig_show(*a,**k)
        _wcplt.show=_wc_show; _wcplt._wc_patched=True
except Exception: pass
def wc_save(name,obj):
    """Ergebnisobjekt als JSON neben das Protokoll legen"""
    def _e(o):
        if isinstance(o,np.ndarray): return o.tolist()
        if isinstance(o,(np.integer,)): return int(o)
        if isinstance(o,(np.floating,)): return float(o)
        if isinstance(o,(np.bool_,)): return bool(o)
        return str(o)
    try:
        with open(os.path.join(RUN_OUT,name+".json"),"w",encoding="utf-8") as f:
            json.dump(obj,f,ensure_ascii=False,indent=1,default=_e)
        print("gespeichert: %s.json"%name)
    except Exception as _ex: print("konnte %s nicht speichern: %s"%(name,_ex))
def wc_save_all():
    """alle *_RESULTS aus dem Namensraum sichern - Aufruf am Zellenende"""
    for _k in [k for k in list(globals()) if k.endswith("_RESULTS")]:
        wc_save(_k,globals()[_k])
    print("Lauf-Ordner:",RUN_OUT)
print("Lauf-Ordner (Protokoll + Abbildungen):",RUN_OUT)
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)

import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import subprocess, sys, time
N_FIT=12; MAXTOK=64; BATCH=6; SEED=0; EPS=0.1; NBOOT=4000
LAYERS=[27,31,35]           # nur die Schichten, in denen der Effekt lebt
L_PRIM=35                   # VORREGISTRIERT: tiefste Schicht, ohne Blick auf Q gewaehlt
N_CTRL=15
MASK_NPZ=(glob.glob("/content/drive/MyDrive/**/vocab_foreign_masks.npz",recursive=True) or [""])[0]
SCAFF="<|im_start|>user\n"
def think_prefix(u,th=""):
    return SCAFF+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n"+th+"\n</think>\n\n"
# ---------------- reine Logik (offline geprueft) ---------------------------
# Locale-Hinweise: nicht-lateinisch -> lateinisch (Arm "latein"),
#                  nicht-lateinisch -> ANDERES nicht-lateinisch (Arm "placebo").
# Der Placebo-Arm aendert gleich viel Text, laesst die geforderte Schrift aber
# fremd. Bricht er mit ein, war es die Bearbeitung und nicht die Schrift.
TAUSCH_LATEIN={
 "Japan":"Brazil","Japanese":"Portuguese","japanese":"portuguese",
 "China":"Germany","Chinese":"German","chinese":"german",
 "Korea":"Poland","Korean":"Polish","korean":"polish",
 "Russia":"Portugal","Russian":"Portuguese","russian":"portuguese",
 "Taiwan":"Austria","Thailand":"Sweden","Thai":"Swedish",
 "India":"Italy","Hindi":"Italian","Israel":"Norway","Hebrew":"Norwegian",
 "Arabic":"Spanish","arabic":"spanish","Greece":"Ireland","Greek":"Irish",
 "Tokyo":"Lisbon","Beijing":"Berlin","Seoul":"Warsaw","Moscow":"Madrid",
 "Baidu":"Dropbox","Naver":"Telenor","Yandex":"Vodafone","WeChat":"WhatsApp",
 "Alibaba":"Ericsson","Rakuten":"Zalando","Tencent":"Spotify","Kakao":"Klarna",
 "Line":"Skype","Weibo":"Twitter","Aliyun":"Hetzner","Mercari":"Etsy",
}
TAUSCH_PLACEBO={
 "Japan":"Korea","Japanese":"Korean","japanese":"korean",
 "China":"Thailand","Chinese":"Thai","chinese":"thai",
 "Korea":"Japan","Korean":"Japanese","korean":"japanese",
 "Russia":"Greece","Russian":"Greek","russian":"greek",
 "Taiwan":"Vietnam","Thailand":"China","Thai":"Chinese",
 "India":"Israel","Hindi":"Hebrew","Israel":"India","Hebrew":"Hindi",
 "Arabic":"Persian","arabic":"persian","Greece":"Russia","Greek":"Russian",
 "Tokyo":"Seoul","Beijing":"Shanghai","Seoul":"Tokyo","Moscow":"Kyiv",
 "Baidu":"Naver","Naver":"Baidu","Yandex":"Weibo","WeChat":"Kakao",
 "Alibaba":"Tencent","Rakuten":"Mercari","Tencent":"Alibaba","Kakao":"WeChat",
 "Line":"Weibo","Weibo":"Line","Aliyun":"Baidu","Mercari":"Rakuten",
}
def tausche(text,tab):
    """EIN Durchgang, alle Schluessel gleichzeitig - nacheinander zu ersetzen
       wuerde bei symmetrischen Tabellen (Japan<->Korea) die Ersetzung wieder
       rueckgaengig machen. Laengste Schluessel zuerst in der Alternative."""
    keys=sorted(tab,key=len,reverse=True)
    pat=re.compile(r"\b(%s)\b"%"|".join(re.escape(k) for k in keys))
    zaehl={}
    def rep(m):
        k=m.group(0); zaehl[k]=zaehl.get(k,0)+1; return tab[k]
    out=pat.sub(rep,text)
    return out,["%s->%s x%d"%(k,tab[k],n) for k,n in sorted(zaehl.items())]
def sample_positions(lo,hi,n,must):
    if hi<=lo: return sorted(set(must))
    step=max(1,(hi-lo)//max(1,n))
    return sorted(set(list(range(lo,hi+1,step))[:n]+[t for t in must if lo<=t<=hi]))
def chunks(xs,k): return [xs[i:i+k] for i in range(0,len(xs),k)]
def rang(vals,idx):
    v=list(vals); t=v[idx]; return 1+sum(1 for x in v if x>t)
def boot_diff(a,b,nboot=4000,seed=0):
    """gepaarter Bootstrap ueber Korpus-Prompts auf log10(a)-log10(b)"""
    a=np.asarray(a,float); b=np.asarray(b,float); n=len(a)
    d=np.log10(np.maximum(a,1e-12))-np.log10(np.maximum(b,1e-12))
    rng=np.random.default_rng(seed)
    bs=np.array([d[rng.integers(0,n,n)].mean() for _ in range(nboot)])
    return float(d.mean()),float(np.percentile(bs,2.5)),float(np.percentile(bs,97.5))
def verdict_tausch(d_lat,lo_lat,hi_lat,d_pla,lo_pla,hi_pla,r1_org,r1_lat,r1_pla,n):
    """d_*: mittleres log10-Verhaeltnis Arm/Original (negativ = Effekt bricht ein)
       r1_*: Anzahl Korpus-Prompts mit Rang 1 fuer Q"""
    lat_bricht = hi_lat<0
    pla_bricht = hi_pla<0
    if lat_bricht and pla_bricht: return "UNSPEZIFISCH"
    if lat_bricht and not pla_bricht: return "KOEDER"
    if not lat_bricht and lo_lat>-0.15: return "LOKALISIERUNG"
    return "UNKLAR"
# ---------------- Umgebung sicherstellen -----------------------------------
JL="/content/jacobian_lens"
if "fd_transport" not in globals():
    if not os.path.isdir(os.path.join(JL,"jlens")):
        r=subprocess.run(["git","clone","--depth","1",
                          "https://github.com/Erikiss/jacobian-lens",JL],
                         capture_output=True,text=True,timeout=600)
        assert r.returncode==0, "Klon fehlgeschlagen"
    subprocess.run([sys.executable,"-m","pip","install","-q","--no-deps","-e",JL],
                   capture_output=True,text=True)
    if JL not in sys.path: sys.path.insert(0,JL)
from jlens.hooks import ActivationRecorder
from jlens.fitting import valid_position_mask
BLOCKS=model.model.layers
for _p in model.parameters(): _p.requires_grad_(False)
def fwd(ids): return model.model(input_ids=ids)
_UD=next(model.model.norm.parameters()).dtype
def unembed(r): return model.lm_head(model.model.norm(r.to(_UD)))
@torch.no_grad()
def fd_transport(input_ids,source_layers,target_layer,V,skip_first,B,eps=EPS):
    """zentrale Differenz - kein Autograd, laeuft durch jeden Kernel"""
    ids=input_ids.expand(B,-1)
    pm=valid_position_mask(ids.shape[1],skip_first=skip_first); npos=int(pm.sum())
    st={"l":None,"sign":0,"tan":None,"pos":pm.nonzero(as_tuple=True)[0]}
    def mk(idx):
        def hook(mod,inp,out):
            if st["l"]!=idx or st["sign"]==0: return None
            t=out if torch.is_tensor(out) else out[0]
            t2=t.clone(); p=st["pos"].to(t2.device)
            t2[:,p,:]=t2[:,p,:]+(st["sign"]*eps)*st["tan"].to(t2.dtype).to(t2.device)[:,None,:]
            return t2 if torch.is_tensor(out) else (t2,)+tuple(out[1:])
        return hook
    hs=[BLOCKS[l].register_forward_hook(mk(l)) for l in source_layers]
    out={}
    try:
        with ActivationRecorder(BLOCKS,at=[target_layer]) as rec:
            def run():
                fwd(ids); a=rec.activations[target_layer]
                return a[:,st["pos"].to(a.device),:].float().sum(1)
            for l in source_layers:
                st["l"]=l; st["tan"]=V[l]
                st["sign"]=1; yp=run(); st["sign"]=-1; ym=run()
                st["l"]=None; st["sign"]=0
                out[l]=(yp-ym)/(2.0*eps*npos)
    finally:
        for h in hs: h.remove()
    return out
# ---------------- Drive-Ausgabe --------------------------------------------
OUT=globals().get("RUN_OUT") or ("/content/drive/MyDrive/WeirdChat_Runs/tausch_"
                                 +time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(OUT,exist_ok=True)
LINES=[]
def P(s=""): LINES.append(str(s))
def schreibe(n,t):
    with open(os.path.join(OUT,n),"w",encoding="utf-8") as f: f.write(t)
FEHLER=None
try:
    # ---------- Arme bauen ---------------------------------------------------
    TAB=PROMPTS[[p for p in PROMPT_IDS if p.startswith("643fdf5d")][0]]
    P("SCHRIFT-TAUSCH - %s"%time.strftime("%Y-%m-%d %H:%M:%S"))
    P("="*72)
    P("ZIELPROMPT im Original (%d Zeichen):"%len(TAB)); P(TAB); P("")
    t_lat,log_lat=tausche(TAB,TAUSCH_LATEIN)
    t_pla,log_pla=tausche(TAB,TAUSCH_PLACEBO)
    P("Ersetzungen latein : %s"%(", ".join(log_lat) if log_lat else "KEINE"))
    P("Ersetzungen placebo: %s"%(", ".join(log_pla) if log_pla else "KEINE"))
    if not log_lat:
        P("")
        P("!! Im Prompt steht kein Locale-Hinweis aus der Tabelle. Der Tausch")
        P("   greift nicht. Der Prompt oben ist gespeichert - der Tausch muss")
        P("   von Hand gebaut werden. Abbruch vor der teuren Rechnung.")
        raise SystemExit
    ARME={"original":TAB,"latein":t_lat,"placebo":t_pla}
    D={}
    P("")
    P("AUSRICHTUNG je Arm:")
    for a,txt in ARME.items():
        pre=think_prefix(txt); e=tokenizer(pre,return_offsets_mapping=True)
        ids=e["input_ids"]; om=e["offset_mapping"]
        c0=len(SCAFF)+txt.index("local name"); c1=c0+len("local name")
        dec=[i for i,(x,y) in enumerate(om) if y>c0 and x<c1 and y>x]
        K,Q=dec[0],dec[-1]
        ut=[i for i,(x,y) in enumerate(om) if y>len(SCAFF) and x<len(SCAFF)+len(txt) and y>x]
        pos=sample_positions(ut[0],ut[-1],N_CTRL,[K-1,K,Q,Q+1])
        D[a]=dict(text=txt,ids=ids,K=K,Q=Q,POS=pos,jQ=pos.index(Q),L=len(ids))
        P("  %-9s %3d Tokens | K=%d %r Q=%d %r | %d Positionen"
          %(a,len(ids),K,tokenizer.decode([ids[K]]),Q,tokenizer.decode([ids[Q]]),len(pos)))
    gleich=len({D[a]["L"] for a in ARME})==1
    gl_tok=len({(tokenizer.decode([D[a]["ids"][D[a]["K"]]]),
                 tokenizer.decode([D[a]["ids"][D[a]["Q"]]])) for a in ARME})==1
    P("  Laenge identisch: %s | Koeder-Tokens identisch: %s"%(gleich,gl_tok))
    if not gl_tok:
        P("  !! Koeder-Tokens unterscheiden sich - der Test waere konfundiert.")
        raise SystemExit
    if not gleich:
        P("  (Laengen weichen ab; Raenge sind arminterne Vergleiche, das ist")
        P("   tolerabel, aber es steht im Bericht.)")
    # ---------- Residuen je Arm ---------------------------------------------
    for a in ARME:
        it=torch.tensor([D[a]["ids"]],device=model.device)
        with torch.no_grad():
            hs=model(input_ids=it,output_hidden_states=True).hidden_states
        D[a]["H"]={l:torch.stack([hs[l+1][0,p] for p in D[a]["POS"]]).float().cpu()
                   for l in LAYERS}
        del hs
    gc.collect(); torch.cuda.empty_cache()
    # ---------- Korpus, gemeinsam fuer alle Arme -----------------------------
    rng=np.random.default_rng(SEED)
    cand=[p for p in PROMPT_IDS if 200<len(PROMPTS[p])<=1200]
    corp=[PROMPTS[cand[i]] for i in rng.permutation(len(cand))[:N_FIT]]
    P(""); P("Korpus: %d Prompts (gemeinsam fuer alle Arme), je bis %d Tokens"
             %(len(corp),MAXTOK))
    TGT=model.config.num_hidden_layers-1
    M_script=torch.tensor(np.load(MASK_NPZ)["script"])
    def fmass(lg):
        p=torch.softmax(lg.float(),-1); V=p.shape[-1]
        m=M_script.to(p.device)
        if m.shape[0]<V: m=torch.cat([m,torch.zeros(V-m.shape[0],dtype=torch.bool,device=m.device)])
        return p[...,m[:V]].sum(-1)
    # PRO Korpus-Prompt auswerten - das ist die Einheit fuer den Bootstrap
    RES={a:{l:[] for l in LAYERS} for a in ARME}      # je Arm/Schicht: Liste (Q/Median, Rang)
    t0=time.time(); nok=0
    for ci,ctext in enumerate(corp):
        cid=tokenizer(ctext,return_tensors="pt",truncation=True,
                      max_length=MAXTOK).input_ids.to(model.device)
        if cid.shape[1]<24: continue
        try:
            for a in ARME:
                pos=D[a]["POS"]; H=D[a]["H"]; jQ=D[a]["jQ"]
                acc={l:torch.zeros(len(pos),model.config.hidden_size) for l in LAYERS}
                for g in chunks(list(range(len(pos))),BATCH):
                    V={l:H[l][g].to(model.device) for l in LAYERS}
                    o=fd_transport(cid,LAYERS,TGT,V,16,len(g))
                    for l in LAYERS: acc[l][g]=o[l].cpu()
                    del o,V
                with torch.no_grad():
                    for l in LAYERS:
                        m=fmass(unembed(acc[l].to(model.device))).cpu().numpy()
                        RES[a][l].append((float(m[jQ]/max(np.median(m),1e-12)),
                                          rang(m,jQ)))
            nok+=1
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); P("  OOM bei Korpus-Prompt %d - uebersprungen"%ci)
        gc.collect(); torch.cuda.empty_cache()
        if (ci+1)%3==0:
            P("  %2d/%d Korpus-Prompts | %.1f min | %.1f s je Prompt"
              %(ci+1,len(corp),(time.time()-t0)/60,(time.time()-t0)/max(ci+1,1)))
    assert nok>=4, "zu wenige Korpus-Prompts durchgelaufen (%d)"%nok
    P("  verwertbar: %d Korpus-Prompts"%nok)
    # ---------- Statistik ----------------------------------------------------
    P(""); P("ERGEBNIS je Arm und Schicht (n=%d Korpus-Prompts)"%nok)
    P("  %-9s %-6s %14s %10s %12s"%("Arm","L","Q/Median (Med)","Rang-1","Rang (Median)"))
    TAB_STAT={}
    for a in ARME:
        for l in LAYERS:
            r=RES[a][l]; ratio=[x[0] for x in r]; rk=[x[1] for x in r]
            TAB_STAT[(a,l)]=dict(ratio=ratio,rank=rk)
            P("  %-9s L%-5d %14.2f %6d/%-3d %12.1f"
              %(a,l,float(np.median(ratio)),sum(1 for x in rk if x==1),len(rk),
                float(np.median(rk))))
    P("")
    P("GEPAARTER BOOTSTRAP auf log10(Q/Median), vorregistrierte Schicht L%d:"%L_PRIM)
    o=TAB_STAT[("original",L_PRIM)]["ratio"]
    dl,ll,hl=boot_diff(TAB_STAT[("latein",L_PRIM)]["ratio"],o,NBOOT,SEED)
    dp,lp,hp=boot_diff(TAB_STAT[("placebo",L_PRIM)]["ratio"],o,NBOOT,SEED)
    P("  latein  - original: %+.3f Dekaden  [%+.3f, %+.3f]"%(dl,ll,hl))
    P("  placebo - original: %+.3f Dekaden  [%+.3f, %+.3f]"%(dp,lp,hp))
    for l in LAYERS:
        if l==L_PRIM: continue
        o2=TAB_STAT[("original",l)]["ratio"]
        d2,l2,h2=boot_diff(TAB_STAT[("latein",l)]["ratio"],o2,NBOOT,SEED)
        d3,l3,h3=boot_diff(TAB_STAT[("placebo",l)]["ratio"],o2,NBOOT,SEED)
        P("  (L%d sekundaer) latein %+.3f [%+.3f,%+.3f] | placebo %+.3f [%+.3f,%+.3f]"
          %(l,d2,l2,h2,d3,l3,h3))
    r1={a:sum(1 for x in TAB_STAT[(a,L_PRIM)]["rank"] if x==1) for a in ARME}
    code=verdict_tausch(dl,ll,hl,dp,lp,hp,r1["original"],r1["latein"],r1["placebo"],nok)
    P(""); P("VERDIKT: %s"%code)
    if code=="KOEDER":
        P("  Der Effekt bricht ein, wenn die geforderte Ausgabeschrift lateinisch")
        P("  wird (%+.2f Dekaden), aber NICHT bei gleich grosser Bearbeitung, die"%dl)
        P("  die Schrift fremd laesst (%+.2f). Die Disposition haengt an der"%dp)
        P("  geforderten SCHRIFT, nicht an der Zeichenkette 'local name'. Die")
        P("  Lokalisierungs-Lesart ist damit ausgeschlossen.")
    elif code=="LOKALISIERUNG":
        P("  Der Effekt ueberlebt den Schrift-Tausch (%+.2f Dekaden, unteres"%dl)
        P("  Ende %+.2f). Was an Q=43 steht, haengt an der Zeichenkette und"%ll)
        P("  nicht an der geforderten Ausgabeschrift - die Koeder-Lesart faellt.")
    elif code=="UNSPEZIFISCH":
        P("  BEIDE Arme brechen ein (latein %+.2f, placebo %+.2f). Es war die"%(dl,dp))
        P("  Bearbeitung selbst, nicht die Schrift. Der Test trennt nicht;")
        P("  es braucht einen Tausch, der noch weniger am Prompt aendert.")
    else:
        P("  Kein klares Bild: latein %+.2f [%+.2f,%+.2f], placebo %+.2f."%(dl,ll,hl,dp))
        P("  Entweder zu wenig Korpus-Prompts oder ein Effekt mittlerer Groesse.")
    P(""); P("(Zielgroesse ist das Verhaeltnis Q zu Median DERSELBEN Schicht und")
    P(" desselben Arms. Absolute Massen sind zwischen Armen nicht vergleichbar.)")
    # ---------- Abbildung ----------------------------------------------------
    fig,axs=plt.subplots(1,len(LAYERS),figsize=(4.6*len(LAYERS),4))
    if len(LAYERS)==1: axs=[axs]
    for ax,l in zip(axs,LAYERS):
        dat=[np.log10(np.maximum(TAB_STAT[(a,l)]["ratio"],1e-12)) for a in ARME]
        ax.boxplot(dat,labels=list(ARME))
        for i,d in enumerate(dat):
            ax.scatter(np.full(len(d),i+1)+np.linspace(-.08,.08,len(d)),d,s=14,
                       color="#DC2626",zorder=3)
        ax.axhline(0,ls="--",c="#888",lw=1)
        ax.set_title("L%d%s"%(l," (vorregistriert)" if l==L_PRIM else ""),fontsize=10)
        ax.set_ylabel("log10(Q / Median)")
    fig.tight_layout(); fig.savefig(os.path.join(OUT,"tausch.png"),dpi=150,bbox_inches="tight")
    TAUSCH_RESULTS=dict(verdict=code,n_corpus=nok,layers=LAYERS,L_prim=L_PRIM,
                        ersetzungen=dict(latein=log_lat,placebo=log_pla),
                        prompts={a:D[a]["text"] for a in ARME},
                        Q={a:D[a]["Q"] for a in ARME},
                        stats={"%s_L%d"%(a,l):TAB_STAT[(a,l)] for a in ARME for l in LAYERS},
                        boot=dict(latein=[dl,ll,hl],placebo=[dp,lp,hp]))
    globals()["TAUSCH_RESULTS"]=TAUSCH_RESULTS
    schreibe("TAUSCH_RESULTS.json",json.dumps(TAUSCH_RESULTS,ensure_ascii=False,indent=1))
except SystemExit:
    pass
except Exception as e:
    import traceback; FEHLER=traceback.format_exc(); P(""); P("ABBRUCH: %s"%e); P(FEHLER)
finally:
    schreibe("bericht_tausch.txt","\n".join(LINES))
    print("GESCHRIEBEN NACH:",OUT)
    print("\n".join(LINES[-45:]) if not FEHLER else FEHLER.splitlines()[-1])
wc_save_all()
